In [16]:
import sqlite3
conn = sqlite3.connect('excel_data.db')
cursor = conn.cursor()
cursor.execute("SELECT filename, result_json FROM files WHERE result_json IS NOT NULL")
rows = cursor.fetchall()
for row in rows:
    print(f"File: {row[0]}")
    print(f"JSON (first 500 chars): {row[1][:500]}...")
conn.close()

File: S2T-700-КЮЛ_v5.xlsx
JSON (first 500 chars): {"filename": "S2T-700-КЮЛ_v5.xlsx", "model_used": "GigaChat-Max", "file_id": "ba3c88f27db04461e7a93941664e9950d0975b5bbd6235230f169a0b215b28d2", "summary": "Этот документ описывает правила преобразования данных из различных источников в систему корпоративного кредитования юридического лица (кодовое название проекта \"КЮЛ\") крупного российского финансового учреждения (вероятно Сбербанка). Он включает спецификации маппинга \"источник-приемник\" (S2T), регламентирующие процессы ETL для передачи ...
File: S2T_308_000046_v0.39.xlsx
JSON (first 500 chars): {"filename": "S2T_308_000046_v0.39.xlsx", "model_used": "GigaChat-Max", "file_id": "16d33af1ba0a5150d09e563aaa02855bcd552336f9316c8e7a89be97f7cccb26", "summary": "Этот документ является спецификацией процесса преобразования данных («источник-приемник», Source-to-Target, S2T) для хранилища данных крупного российского банка, посвящённого бизнес-домену \"Эскроу\". Он охватывает все этапы движ

In [17]:
import sqlite3
import pandas as pd
import json
conn = sqlite3.connect('excel_data.db')
cursor = conn.cursor()
file_id = '984a58c7cf9cbcd841e8dc0832450da3f19b5083cfa46741cc046cb978c2c1ad'
cursor.execute("SELECT result_json FROM files WHERE file_id = ?", (file_id,))
row = cursor.fetchone()
if row:
    stored_json = row[0]
    # parse back to dict if needed
    data = json.loads(stored_json)
conn.close()

In [22]:
for sheet in data['sheets']:
    print(sheet['sheet_name'], '->', sheet['skipped'])

pxf2a -> True
a2b_columns -> False
source_tables -> True
source_columns -> False
target_tables -> True
target_columns -> False
s2t -> False
additional_objects -> False
change_log -> True
s305_0999_enums -> True


In [14]:
import json
from old.schema_matcher import compare_with_target

excel_json = data

report = compare_with_target(excel_json)

print(f"Similarity score: {report['similarity_score']}%")
print("\nMapping suggestions:")
for sug in report["mapping_suggestions"]:
    if sug["target_table"]:
        print(f"  Sheet '{sug['excel_sheet']}' -> Table '{sug['target_table']}' (similarity: {sug['similarity']})")
        print(f"    Column mapping: {sug['column_mapping']}")
    else:
        print(f"  Sheet '{sug['excel_sheet']}' -> No match ({sug['explanation']})")

Similarity score: 90%

Mapping suggestions:
  Sheet 'pxf2a' -> No match (Нет подходящих столбцов для сопоставления ни с одной из таблиц целевой схемы.)
  Sheet 'a2b_columns' -> Table 'column_mappings' (similarity: high)
    Column mapping: {'target_table_name': 'Название приемника', 'target_column': 'Название поля приемника', 'source_table_name': 'Название источника', 'source_column': 'Название поля источника', 'transformation_rule': None, 'data_type': 'Тип данных поля приемника', 'is_primary_key': 'Первичный ключ приемника'}
  Sheet 'source_tables' -> Table 'source_tables' (similarity: high)
    Column mapping: {'name': 'Имя таблицы', 'description': 'Описание таблицы', 'system_code': None}
  Sheet 'source_columns' -> Table 'column_mappings' (similarity: medium)
    Column mapping: {'target_table_name': 'table', 'target_column': 'column', 'source_table_name': None, 'source_column': None, 'transformation_rule': 'VLOOKUP("t_core_"&MID([@table]; 16; 100)&"."&[@column]; pdm_table[[PDM_KEY]

In [9]:
conn = sqlite3.connect('excel_data.db')
cursor = conn.cursor()
files_df = pd.read_sql_query("SELECT file_id FROM files LIMIT 2", conn)
file_id = files_df.file_id.values[1]
cursor.execute("SELECT result_json FROM files WHERE file_id = ?", (file_id,))
row = cursor.fetchone()
if row:
    stored_json = row[0]
    # parse back to dict if needed
    data = json.loads(stored_json)
conn.close()

In [7]:
excel_json = data

report = compare_with_target(excel_json)

print(f"Similarity score: {report['similarity_score']}%")
print("\nMapping suggestions:")
for sug in report["mapping_suggestions"]:
    if sug["target_table"]:
        print(f"  Sheet '{sug['excel_sheet']}' -> Table '{sug['target_table']}' (similarity: {sug['similarity']})")
        print(f"    Column mapping: {sug['column_mapping']}")
    else:
        print(f"  Sheet '{sug['excel_sheet']}' -> No match ({sug['explanation']})")

Similarity score: 89%

Mapping suggestions:
  Sheet 'История изменений документа' -> No match (Нет соответствующей таблицы для хранения информации об изменениях документов.)
  Sheet 'S2T' -> Table 'column_mappings' (similarity: high)
    Column mapping: {'target_table_name': 'Приемник данных > Целевая таблица', 'target_column': 'Приемник данных > Поле приемника данных', 'source_table_name': 'Источник данных > Название таблицы-источника (из В-таблицы)', 'source_column': 'Источник данных > Поле источника данных', 'transformation_rule': 'Трансформация > Правило трансформации', 'data_type': 'Приемник данных > Тип данных', 'is_primary_key': 'Приемник данных > Первичный ключ'}
  Sheet 'Additional objects' -> Table 'additions' (similarity: high)
    Column mapping: {'table_name': 'Название объекта', 'table_description': 'Описание', 'source_tables_name': 'Таблицы-источники', 'sql': 'SQL', 'description': 'Комментарий'}
  Sheet 'Source tables' -> Table 'source_tables' (similarity: high)
    Colu

In [10]:
file_id

'ba3c88f27db04461e7a93941664e9950d0975b5bbd6235230f169a0b215b28d2'